---
title: "Career Tenure in the NHL: The Full Analysis"
format:
  html:
    code-fold: true
    code-tools: true
    embed-resources: true
---

# Career Tenure in the NHL: An Exploratory Analysis

This notebook is the long version of the career-tenure deep dive. The post makes
four claims; this notebook tests them, tries to break them, and reports what
survived.

The central question is **whether experience — measured in NHL seasons rather
than birthdays — tells us anything about how teams win.** Along the way we have
to answer a prior question that turned out to be more interesting: the NHL is
more veteran-heavy than at any point in its history, and it is not obvious why.

We come at it from several angles:

- Is the league really more experienced than ever, and on which measure?
- Are careers getting longer? (Spoiler: no, and this took three attempts to get right.)
- If not, where do the veterans come from?
- Do more experienced teams win more games, and does that survive controls?
- Do they win the Stanley Cup?
- Does any of this look different for the great players?

## Data Notes

Everything comes from the NHL Stats API (`api.nhle.com/stats/rest/en`), pulled
per season with `cayenneExp = "seasonId={season} and gameTypeId=2"`. Regular
season only unless stated. Twelve tables live in
`s3://hockey-decoded/static-ds-analyses/career-tenure/` and the notebook reads
them directly, so it is runnable as-is with AWS credentials.

Two construction decisions carry most of the weight and are worth stating before
any analysis:

**Experience is prior NHL seasons, not career length.** For a player in season
*Y* who debuted in *D*, experience is `Y - D`; a rookie is 0. Career length is
unknowable at the time and is mechanically correlated with being good, which is
the exact confound the analysis is trying to separate out.

**Team experience is weighted by games played for that team.** A roster is the
men who dressed, not the forty who passed through. Unweighted means are badly
distorted in the small-roster eras.

One data-engineering note that matters: the season-wide summary endpoint returns
a traded player as a single row with combined games, so his experience cannot be
attributed to either club. Adding `teamId` to the filter splits him correctly, at
a cost of ~4,100 requests instead of ~220. 4,377 of 57,186 player-team-seasons
are affected.

In [ ]:
# ----------------------------------------------------------------------
# SETUP
# ----------------------------------------------------------------------
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from statsmodels.duration.survfunc import SurvfuncRight
from statsmodels.nonparametric.smoothers_lowess import lowess

import warnings
warnings.filterwarnings("ignore")

# Brand styling, shared with the figures in the post
HERE = Path("/Users/dwiwad/dev/hockey_site/scripts/career-player-tenure")
if str(HERE) not in sys.path:
    sys.path.insert(0, str(HERE))
from hd_style import (COL_BLUE_DARK as BLUE_DARK, COL_BLUE_PALE as BLUE_PALE,
                      COL_OIL_ORANGE as ORANGE, COL_GRAY as GRAY)

PREFIX = "s3://hockey-decoded/static-ds-analyses/career-tenure"
SO = {"anon": False}

def read(name):
    return pd.read_parquet(f"{PREFIX}/{name}.parquet", storage_options=SO)

pd.set_option("display.width", 140)
print("Setup complete.")

In [ ]:
# ----------------------------------------------------------------------
# LOAD
# ----------------------------------------------------------------------
player_seasons = read("player_seasons")     # one row per player-season
player_careers = read("player_careers")     # one row per player
team_rosters   = read("team_rosters")       # one row per player-team-season
team_seasons   = read("team_seasons")       # standings
star_index     = read("star_index")         # early-career scoring
skater_bios    = read("skater_bios")        # birth date, draft, HOF
playoffs       = read("playoff_team_seasons")

for n, d in [("player_seasons", player_seasons), ("player_careers", player_careers),
             ("team_rosters", team_rosters), ("team_seasons", team_seasons),
             ("star_index", star_index), ("playoffs", playoffs)]:
    print(f"{n:<16} {len(d):>7,} rows")

for d in (player_seasons, team_rosters, team_seasons, playoffs):
    if "season" in d:
        d["yr"] = d.season // 10000

print(f"\nseasons {player_seasons.yr.min()}-{player_seasons.yr.max()}, "
      f"{player_seasons.yr.nunique()} played "
      f"(2004-05 cancelled, so 108 not 109)")

## Constructing the Measures

Three derived quantities used throughout.

**Tenure (`ten`)** — a player's season count, inclusive, so a rookie is 1.
Built from an *ordinal* season index rather than calendar years, so the cancelled
2004-05 season consumes no index. A career running 2003-04 to 2005-06 is two
seasons, not three. Getting this wrong silently adds a phantom season to
everyone active in that era.

**Career span** — `last_index - first_index + 1`. Counts gap years as tenure,
which is deliberate: it measures how long someone lasted in the league, not how
many years they suited up. It diverges from seasons-played for about a quarter
of players.

**Censoring** — anyone appearing in the final observed season has an unfinished
career. They are right-censored throughout, never dropped. Dropping them is the
single most common way to get this analysis wrong, and we demonstrate the damage
later.

In [ ]:
# ----------------------------------------------------------------------
# DERIVED MEASURES
# ----------------------------------------------------------------------
ps = player_seasons.merge(
    player_careers[["playerId", "first_idx", "first_year", "last_year", "span", "censored"]],
    on="playerId")

# ordinal season index: 2004-05 consumes no index
order = {y: i for i, y in enumerate(np.sort(ps.yr.unique()))}
ps["sidx"] = ps.yr.map(order)
ps["ten"] = ps.sidx - ps.first_idx + 1      # inclusive: rookie = 1
ps["exp"] = ps.ten - 1                       # prior seasons: rookie = 0

LAST = int(ps.yr.max())
print(f"lockout absent from the index: {20042005 not in set(player_seasons.season)}")
print(f"tenure range {ps.ten.min()}-{ps.ten.max()}, no negatives: {(ps.ten >= 1).all()}")

# team-season experience, games-weighted
tr = team_rosters.copy()
tr["exp"] = tr.yr - tr.groupby("playerId").yr.transform("min")
tr = tr[tr.gp > 0]

def team_metrics(g):
    w, e = g.gp.to_numpy(float), g.exp.to_numpy(float)
    mu = np.average(e, weights=w)
    return pd.Series({"exp": mu,
                      "exp_sd": np.sqrt(np.average((e - mu) ** 2, weights=w)),
                      "rookie_share": w[e == 0].sum() / w.sum(),
                      "vet_share": w[e >= 9].sum() / w.sum()})

tm = (tr.groupby(["season", "teamId"]).apply(team_metrics, include_groups=False)
        .reset_index())
TS = team_seasons.merge(tm, on=["season", "teamId"]).query("gp >= 20").copy()
for c in ["exp", "point_pct", "rookie_share", "vet_share"]:
    TS[c + "_d"] = TS[c] - TS.groupby("season")[c].transform("mean")
TS["franchise_age"] = TS.yr - TS.groupby("teamId").yr.transform("min")

print(f"\n{len(TS):,} team-seasons, {TS.yr.nunique()} seasons, {TS.teamId.nunique()} franchises")

## Descriptives: What Does a Career Look Like?

Before any test, the shape of the thing. Career length is heavily right-skewed
and lumpy, which is why mean and median tell different stories and why several
of the results below hinge on which one you quote.

Active players are excluded here — their careers are unfinished, not short.

In [ ]:
# ----------------------------------------------------------------------
# CAREER LENGTH DISTRIBUTION
# ----------------------------------------------------------------------
done = player_careers[~player_careers.censored]
print("=" * 62)
print("CAREER LENGTH (completed careers only)")
print("=" * 62)
print(done.span.describe().round(2).to_string())
print(f"\nskew {done.span.skew():.2f}")
print(f"exactly one season : {(done.span == 1).mean()*100:.1f}%")
print(f"10 or more seasons : {(done.span >= 10).mean()*100:.1f}%")
print(f"15 or more seasons : {(done.span >= 15).mean()*100:.1f}%")
print("\npercentiles: " + "  ".join(
    f"p{int(p*100)}={done.span.quantile(p):.0f}" for p in [.1,.25,.5,.75,.9,.95,.99]))

gap = done.span - done.n_seasons
print(f"\nspan vs seasons actually played: differ for {(gap>0).mean()*100:.0f}% "
      f"of players, mean gap {gap.mean():.2f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, col, lbl in [(axes[0], "span", "Seasons elapsed"),
                     (axes[1], "n_seasons", "Seasons played")]:
    v = done[col]
    ax.hist(v, bins=np.arange(0.5, 26.5, 1), color=BLUE_PALE, edgecolor="white", lw=.6)
    ax.axvline(v.median(), color=BLUE_DARK, lw=2.4, label=f"median {v.median():.0f}")
    ax.axvline(v.mean(), color=ORANGE, lw=2.4, ls="--", label=f"mean {v.mean():.1f}")
    ax.set_xlabel(lbl); ax.set_ylabel("Players"); ax.legend()
    sns.despine(ax=ax); ax.grid(False)
plt.tight_layout(); plt.show()

## Hypothesis 1: The League Is More Veteran Than Ever

**Null**: the experience composition of the league today is unremarkable
relative to its history.

**Alternative**: the veteran share is at or near a historic high.

This is descriptive rather than inferential, but the measure matters more than
you would expect — the answer flips depending on where you draw the veteran line.

In [ ]:
# ----------------------------------------------------------------------
# H1: LEAGUE COMPOSITION OVER TIME
# ----------------------------------------------------------------------
comp = ps.groupby("yr").apply(lambda g: pd.Series({
    "players": g.playerId.nunique(),
    "rookie":  (g.ten == 1).mean(),
    "vet10":   (g.ten >= 10).mean(),
    "vet11":   (g.ten >= 11).mean(),
    "mean_ten": g.ten.mean(),
}), include_groups=False)

print("THRESHOLD SENSITIVITY -- the headline depends on where the line is drawn")
for col, lbl in [("vet10", "10th season or later"), ("vet11", "11th season or later")]:
    v = comp[col]
    print(f"\n  {lbl}")
    print(f"    {LAST}-{str(LAST+1)[-2:]}: {v.loc[LAST]*100:.1f}%   "
          f"rank {(v > v.loc[LAST]).sum()+1} of {len(v)}")
    print(f"    top 3 ever: " + ", ".join(f"{y} {x*100:.1f}%" for y, x in v.nlargest(3).items()))

print("\n\nBy decade:")
print(f"{'decade':>8} {'rookie':>8} {'10th+':>8} {'mean tenure':>13}")
for lo in range(1930, 2030, 10):
    w = comp.loc[lo:lo+9]
    if len(w):
        print(f"{lo}s {w.rookie.mean()*100:>7.1f}% {w.vet10.mean()*100:>7.1f}% "
              f"{w.mean_ten.mean():>12.2f}")

**Finding.** On the 10th-season threshold, 2025-26 is the most veteran season in
NHL history. On the 11th-season threshold it is second, behind 1966-67 — the last
season before expansion doubled the league.

Both definitions put the current league in the top two ever, and in both the only
competitor is the final Original Six season. That is the robust claim; a single
threshold is not.

## Hypothesis 2: Careers Are Getting Longer

**Null**: career length has not changed systematically over NHL history.

**Alternative**: careers are lengthening, which would explain the veteran share.

This is the obvious explanation for H1 and it is wrong. It is also where the
analysis is easiest to get backwards, because the answer depends on the baseline
decade you pick and on whether you handle censoring.

In [ ]:
# ----------------------------------------------------------------------
# H2: CAREER LENGTH BY DEBUT COHORT
# ----------------------------------------------------------------------
pc = player_careers.copy()
pc["decade"] = (pc.first_year // 10) * 10

rows = []
with np.errstate(divide="ignore", invalid="ignore"):
    for dec, g in pc.groupby("decade"):
        if len(g) < 150 or dec < 1930 or dec > 2000:
            continue
        sf = SurvfuncRight(g.span.values, (~g.censored).astype(int).values)
        def S(t):
            m = sf.surv_times <= t
            return sf.surv_prob[m][-1] if m.any() else 1.0
        row = {"decade": int(dec), "n": len(g)}
        for p in [.25, .50, .75, .90]:
            try: row[f"p{int(p*100)}"] = sf.quantile(p)
            except Exception: row[f"p{int(p*100)}"] = np.nan
        # mean career = area under the survival curve, censoring-aware
        row["mean"] = sum(S(t) for t in range(26))
        row["reach10"] = S(10)
        rows.append(row)
COH = pd.DataFrame(rows).set_index("decade")
print("CAREER LENGTH BY DEBUT DECADE (Kaplan-Meier)")
print(COH.round(2).to_string())

x = COH.index.values.astype(float)
for c in ["p50", "mean"]:
    m = sm.OLS(COH[c].values, sm.add_constant(x)).fit()
    print(f"\ntrend in {c}: {m.params[1]*10:+.3f} seasons per decade, p={m.pvalues[1]:.3f}")

**Finding: null.** Neither the median nor the mean trends. The 1960s produced the
longest careers of any decade and the 1970s the shortest — an expansion effect,
since new franchises absorb marginal players who wash out quickly.

This is worth dwelling on because it is a trap. Anchor on the 1970s and every
subsequent decade looks like growth. Anchor on the 1960s and nothing does. The
honest summary is cyclical variation with no direction.

## Hypothesis 3: The Veteran Share Is Driven by Falling Intake

If careers have not changed, the composition shift has to come from the flow of
players through the league rather than their durability.

**Null**: intake dynamics cannot account for the rising veteran share.

**Alternative**: they can — a population that admits proportionally fewer new
members ages, regardless of how long anyone stays.

The test is a simulation. Take **one unchanging survival curve**, so careers
cannot change by construction, feed it the actual debut counts each year, and see
how much of the observed veteran share it reproduces.

In [ ]:
# ----------------------------------------------------------------------
# H3a: THE FLOWS
# ----------------------------------------------------------------------
ps["deb"] = ps.yr == ps.first_year
ps["ext"] = ps.yr == ps.last_year
FL = ps.groupby("yr").agg(L=("playerId", "nunique"), debuts=("deb", "sum"),
                          exits=("ext", "sum"))
FL["teams"] = team_seasons.groupby("yr").teamId.nunique()
FL = FL.dropna()
FL["in_pct"] = FL.debuts / FL.L
FL["out_pct"] = FL.exits / FL.L

print("FLOWS THROUGH THE LEAGUE")
print(f"{'period':>11} {'teams':>6} {'players':>8} {'debuts':>7} {'in %':>7} {'out %':>7}")
for lo, hi in [(1942,1966),(1967,1979),(1980,1990),(1991,2000),
               (2001,2010),(2011,2020),(2021,LAST-1)]:
    w = FL.loc[lo:hi]
    print(f"{lo}-{hi} {w.teams.mean():>6.1f} {w.L.mean():>8.0f} {w.debuts.mean():>7.0f} "
          f"{w.debuts.sum()/w.L.sum()*100:>6.1f}% {w.exits.sum()/w.L.sum()*100:>6.1f}%")

print("\nGrowth ratios depend entirely on the baseline -- state which you use:")
now_L, now_d = FL.loc[2020:, "L"].mean(), FL.loc[2020:, "debuts"].mean()
for a, b, lbl in [(1942,1966,"Original Six"),(1960,1969,"1960s"),
                  (1970,1979,"1970s"),(1980,1989,"1980s")]:
    bL, bd = FL.loc[a:b, "L"].mean(), FL.loc[a:b, "debuts"].mean()
    print(f"  from {lbl:<13} league {now_L/bL:.2f}x   debuts {now_d/bd:.2f}x")

In [ ]:
# ----------------------------------------------------------------------
# H3b: SIMULATION WITH CAREERS FROZEN
# ----------------------------------------------------------------------
pool = pc[(~pc.censored) & (pc.first_year.between(1950, 2000))]
sf = SurvfuncRight(pool.span.values, np.ones(len(pool), int))
S = np.array([sf.surv_prob[sf.surv_times <= t][-1] if (sf.surv_times <= t).any() else 1.0
              for t in range(31)])
print(f"fixed survival curve from {len(pool):,} completed careers, 1950-2000 debuts")
print(f"  mean career {S.sum():.2f} seasons")
print(f"  steady-state veteran share this curve implies: {S[10:].sum()/S.sum()*100:.1f}%")
print("  (in equilibrium, composition depends ONLY on survival - entry sets size, not age)\n")

years = [y for y in FL.index]
sim = {}
for y in years:
    tot = vet = 0.0
    for cy in years:
        if cy > y: break
        t = y - cy
        if t < len(S) - 1:
            alive = FL.loc[cy, "debuts"] * S[t]
            tot += alive
            if t >= 9: vet += alive
    sim[y] = vet / tot if tot else np.nan
SIM = pd.Series(sim)

print("OBSERVED vs SIMULATED veteran share (careers frozen by construction)")
print(f"{'period':>11} {'observed':>9} {'simulated':>10}")
for lo, hi in [(1967,1979),(1980,1990),(1991,2000),(2001,2010),(2011,2020),(2021,LAST)]:
    print(f"{lo}-{hi} {comp.vet10.loc[lo:hi].mean()*100:>8.1f}% {SIM.loc[lo:hi].mean()*100:>9.1f}%")
print(f"\n{LAST}-{str(LAST+1)[-2:]}: observed {comp.vet10.loc[LAST]*100:.1f}%   "
      f"simulated {SIM.loc[LAST]*100:.1f}%")
print(f"\nDecomposition of today's {comp.vet10.loc[LAST]*100:.1f}%:")
print(f"  {S[10:].sum()/S.sum()*100:5.1f}%  equilibrium implied by the survival curve")
print(f"  {SIM.loc[LAST]*100:5.1f}%  after layering on the actual (falling) intake")
print(f"  {comp.vet10.loc[LAST]*100:5.1f}%  observed")

In [ ]:
# ----------------------------------------------------------------------
# H3c: THE RESIDUAL -- DID VETERANS STOP LEAVING?
# ----------------------------------------------------------------------
# Year-over-year retention. The successor season is looked up in the data, NOT
# computed as yr+1: 2004-05 does not exist, and treating it as a season away
# scores every 2003-04 player as having left.
act = {y: set(g.playerId) for y, g in ps.groupby("yr")}
Y = sorted(act)
nxt = {y: Y[i+1] for i, y in enumerate(Y[:-1])}
ret = ps[ps.yr.isin(nxt)].copy()
ret["ret"] = [p in act[nxt[y]] for p, y in zip(ret.playerId, ret.yr)]
ret["grp"] = pd.cut(ret.ten, [0,1,4,9,99],
                    labels=["Rookie","2nd-4th","5th-9th","10th+"])
R = ret.groupby(["yr","grp"], observed=True).ret.mean().unstack()

print("RETENTION BY TENURE GROUP (share returning next season)")
print(f"{'decade':>8} " + "".join(f"{c:>10}" for c in R.columns))
for lo in [1950,1970,1980,1990,2000,2010,2020]:
    w = R.loc[lo:lo+9].mean()
    print(f"{lo}s " + "".join(f"{v*100:>9.1f}%" for v in w))

print("\nTrend since 1970:")
for c in R.columns:
    w = R.loc[1970:, c].dropna()
    m = sm.OLS(w.values, sm.add_constant(w.index.values.astype(float))).fit()
    print(f"  {str(c):>8}: {m.params[1]*1000:+.2f} pp per decade   p={m.pvalues[1]:.4f}")

**Finding: partly.** Frozen careers plus real intake reproduces the observed
veteran share within a point or two through 2020, so for that whole period the
composition shift is pure arithmetic. For the most recent seasons the simulation
undershoots by several points.

The residual has a candidate. Retention rises significantly, and it rises
**only among veterans** — rookie retention is statistically flat while
tenth-season-plus players became markedly more likely to return. Long careers got
a little longer at the far end; short ones did not change at all. That lifts the
mean weakly and the median not at all, which is exactly the pattern in H2.

So the honest answer is roughly half intake arithmetic, half veterans staying.

**Caveat.** The simulation uses one pooled survival curve for every cohort, so it
cannot capture expansion-era swings, and it ignores returnees. Treat it as
indicative of the modern era, not a formal decomposition of the century.

## Hypothesis 4: Experienced Teams Win More

**Null**: team experience is unrelated to winning.

**Alternative**: more experienced teams win more.

The difficulty is that good players last longer, so an experienced roster is
partly just a good roster. Rather than adjust that away, we ask the same question
at three levels of comparison, each holding more constant.

In [ ]:
# ----------------------------------------------------------------------
# H4a: SEASON LEVEL, FOUR SPECIFICATIONS
# ----------------------------------------------------------------------
def fit(y, X, groups):
    return sm.OLS(y, sm.add_constant(X)).fit(cov_type="cluster",
                                             cov_kwds={"groups": groups})

seas_d = pd.get_dummies(TS.season, prefix="s", drop_first=True).astype(float)
fran_d = pd.get_dummies(TS.teamId, prefix="t", drop_first=True).astype(float)

specs = {}
specs["Raw"] = fit(TS.point_pct, TS[["exp"]], TS.season)
specs["+ season FE"] = fit(TS.point_pct, pd.concat([TS[["exp"]], seas_d], axis=1), TS.season)
specs["+ franchise FE"] = fit(TS.point_pct,
                              pd.concat([TS[["exp"]], seas_d, fran_d], axis=1), TS.season)
d2 = TS.sort_values(["teamId", "yr"])
lag = d2.groupby("teamId")[["yr", "exp", "point_pct"]].shift(1)
fd = (d2.assign(d_exp=d2.exp - lag.exp, d_pp=d2.point_pct - lag.point_pct,
                gap=d2.yr - lag.yr).query("gap == 1").dropna(subset=["d_exp", "d_pp"]))
specs["Year-on-year, same team"] = fit(fd.d_pp, fd[["d_exp"]], fd.season)

print("POINTS PERCENTAGE PER EXTRA SEASON OF TEAM EXPERIENCE")
print(f"{'specification':>26} {'coef (pp)':>10} {'SE':>6} {'t':>7} {'n':>7}")
for name, m in specs.items():
    k = "d_exp" if "d_exp" in m.params.index else "exp"
    print(f"{name:>26} {m.params[k]*100:>+10.2f} {m.bse[k]*100:>6.2f} "
          f"{m.tvalues[k]:>+7.1f} {int(m.nobs):>7,}")

In [ ]:
# ----------------------------------------------------------------------
# H4b: GAME LEVEL -- BETWEEN vs WITHIN TEAM-SEASON
# ----------------------------------------------------------------------
# This is the decisive test. Within a team-season the roster is essentially
# fixed; what varies is which 20 players dressed on a given night.
L = read("team_games")
ts_id = L.teamId.astype(str) + "_" + L.season.astype(str)

def cl(y, X):
    return sm.OLS(y, sm.add_constant(X)).fit(cov_type="cluster",
                                             cov_kwds={"groups": ts_id})

B = cl(L.won, L[["between"]]); W = cl(L.won, L[["within"]])
dm = L[["won","exp","opp_exp","home"]].sub(
    L.groupby(["teamId","season"])[["won","exp","opp_exp","home"]].transform("mean"))
# `within` is already demeaned, so by Frisch-Waugh this IS the FE estimator
assert abs(cl(dm.won, dm[["exp"]]).params["exp"] - W.params["within"]) < 1e-6
fe = cl(dm.won, dm[["exp","opp_exp","home"]])

print(f"{len(L):,} team-games, {L.game_id.nunique():,} games, "
      f"{L.groupby(['teamId','season']).ngroups} team-seasons\n")
print(f"  BETWEEN teams   {B.params['between']*100:+.2f} pp/season  "
      f"(SE {B.bse['between']*100:.2f})")
w, wse = W.params["within"], W.bse["within"]
print(f"  WITHIN a team   {w*100:+.2f} pp/season  (SE {wse*100:.2f}), "
      f"95% CI [{(w-1.96*wse)*100:+.2f}, {(w+1.96*wse)*100:+.2f}]")
print("\nfull model with team-season FE, opponent experience and home ice:")
print(fe.summary2().tables[1].round(4).to_string())

**Finding: split.** Between teams, a season of extra experience is worth about
three points of points percentage. Within a team-season it is worth nothing, and
the interval is tight enough to rule out anything larger than about one point.

Note the coherence check in the full model: the opponent's experience coefficient
is almost exactly the mirror of the own-experience coefficient. The model does not
know those should match; that they do is evidence the measure is capturing
something real about team strength.

**Interpretation.** Experience marks a good roster. We cannot show it acts on the
game. The within-team null is also conservative in an interesting way — a call-up
replacing an injured veteran is usually both less experienced *and* worse, so the
test bundles an experience downgrade with a quality downgrade and still finds
nothing.

## Hypothesis 5: Experienced Teams Win the Stanley Cup

**Null**: among teams reaching the final, experience does not predict the winner.

**Alternative**: the more experienced finalist wins more often.

Cup winners are identified as the team with the most playoff wins that season,
validated against the historical record. Four early seasons tie on wins; two of
those resolve incorrectly and are corrected by hand.

In [ ]:
# ----------------------------------------------------------------------
# H5: POSTSEASON OUTCOMES
# ----------------------------------------------------------------------
PL = playoffs.copy()
PL["rk"] = PL.groupby("yr").w.rank(ascending=False, method="min")
win = PL[PL.rk == 1].drop_duplicates("yr")[["yr","teamId"]].assign(result="Won the Cup")
# 1917-18 and 1927-28 tie on playoff wins and resolve to the wrong team
for _y, _n in {1917: "Toronto Arenas", 1927: "New York Rangers"}.items():
    _t = PL.loc[(PL.yr == _y) & (PL.team == _n), "teamId"]
    if len(_t): win.loc[win.yr == _y, "teamId"] = _t.iloc[0]
run = PL[PL.rk == 2].drop_duplicates("yr")[["yr","teamId"]].assign(result="Lost the final")

D = TS.merge(PL[["yr","teamId"]].assign(playoff=1), on=["yr","teamId"], how="left")
D["playoff"] = D.playoff.fillna(0).astype(int)
D = D.merge(pd.concat([win, run]), on=["yr","teamId"], how="left")
D["result"] = D.result.where(D.result.notna(), pd.Series(
    np.where(D.playoff == 1, "Made playoffs", "Missed playoffs"), index=D.index))
D = D[D.yr <= LAST - 1]
lg = D.groupby("season").exp.mean().rename("league")
D = D.merge(lg, on="season")

ORDER = ["Won the Cup","Lost the final","Made playoffs","Missed playoffs"]
print("EXPERIENCE BY POSTSEASON OUTCOME")
print(f"{'outcome':>18} {'n':>5} {'raw':>6} {'league then':>12} {'vs league':>10} {'% above':>9}")
for r in ORDER:
    g = D[D.result == r]
    print(f"{r:>18} {len(g):>5} {g.exp.mean():>6.2f} {g.league.mean():>12.2f} "
          f"{g.exp_d.mean():>+10.2f} {(g.exp_d > 0).mean()*100:>8.0f}%")

fin = (D[D.result=="Won the Cup"].set_index("yr").exp_d.rename("win").to_frame()
       .join(D[D.result=="Lost the final"].set_index("yr").exp_d.rename("lose"), how="inner"))
t = sm.stats.ttest_ind(fin.win, fin.lose, usevar="unequal")
print(f"\nFINALS, paired ({len(fin)} finals):")
print(f"  winner {fin.win.mean():+.3f}   loser {fin.lose.mean():+.3f}   "
      f"diff {fin.win.mean()-fin.lose.mean():+.3f}")
print(f"  winner more experienced in {(fin.win > fin.lose).mean()*100:.0f}% of finals")
print(f"  t = {t[0]:+.2f}, p = {t[1]:.3f}")

m = sm.Logit(D.playoff, sm.add_constant(D[["exp_d"]])).fit(disp=0)
print(f"\nMAKING the playoffs: logit coef {m.params.exp_d:+.4f}, z = {m.tvalues.exp_d:+.1f}")

**Finding: null at the Cup, strong at the playoff line.** Teams that qualified
were meaningfully more experienced than teams that missed. Between the two teams
that actually reached the final, experience is a coin flip.

Note the raw column next to the league column. Cup winners have a *lower* raw
mean than teams that merely made the playoffs — an era artifact, since there is
one champion per season whatever the league size, so finalists spread evenly
across the century while playoff teams concentrate in the 32-team era. Raw
tenure cannot be pooled across eras; everything here is measured against each
season's own league.

## Robustness

Five ways the results above could be artifacts, and what happens when you check.

In [ ]:
# ----------------------------------------------------------------------
# R1: DOES THE WINNING RESULT DEPEND ON INCLUDING GOALIES?
# ----------------------------------------------------------------------
# Goalies are ~4% of the games-weighted team mean but are excluded from the
# star index entirely, so it is worth knowing whether they drive anything.
for lbl, sub in [("all players", tr), ("skaters only", tr[tr.position != "G"])]:
    t2 = (sub.groupby(["season","teamId"])
            .apply(lambda g: pd.Series({"exp": np.average(g.exp, weights=g.gp)}),
                   include_groups=False).reset_index())
    d3 = team_seasons.merge(t2, on=["season","teamId"]).query("gp >= 20").copy()
    for c in ["exp","point_pct"]:
        d3[c+"_d"] = d3[c] - d3.groupby("season")[c].transform("mean")
    m = sm.OLS(d3.point_pct_d, sm.add_constant(d3[["exp_d"]])).fit(
        cov_type="cluster", cov_kwds={"groups": d3.season})
    print(f"  {lbl:<14} coef {m.params.exp_d*100:+.2f} pp   t {m.tvalues.exp_d:+.1f}   "
          f"r {d3.exp_d.corr(d3.point_pct_d):+.3f}")
print("\n-> survives; goalies are not driving the result")

In [ ]:
# ----------------------------------------------------------------------
# R2: IS "EXPERIENCE" JUST AGE?
# ----------------------------------------------------------------------
b = skater_bios[["playerId","birth_date"]].copy()
b["birth"] = pd.to_datetime(b.birth_date, errors="coerce")
sk = tr[tr.position != "G"].merge(b, on="playerId")
sk["age"] = (pd.to_datetime(sk.yr.astype(str) + "-10-01") - sk.birth).dt.days / 365.25
sk = sk[sk.age.between(16, 45)]
ta = (sk.groupby(["season","teamId"]).apply(lambda g: pd.Series({
        "age": np.average(g.age, weights=g.gp),
        "exp": np.average(g.exp, weights=g.gp)}), include_groups=False).reset_index())
A = team_seasons.merge(ta, on=["season","teamId"]).query("gp >= 20").copy()
for c in ["age","exp","point_pct"]:
    A[c+"_d"] = A[c] - A.groupby("season")[c].transform("mean")

print(f"corr(team age, team experience) within season: {A.age_d.corr(A.exp_d):+.3f}\n")
print("separately:")
for v in ["exp","age"]:
    m = sm.OLS(A.point_pct_d, sm.add_constant(A[[v+"_d"]])).fit(
        cov_type="cluster", cov_kwds={"groups": A.season})
    print(f"  {v:<4} {m.params[v+'_d']*100:+6.2f} pp   t {m.tvalues[v+'_d']:+5.1f}")
m = sm.OLS(A.point_pct_d, sm.add_constant(A[["exp_d","age_d"]])).fit(
    cov_type="cluster", cov_kwds={"groups": A.season})
print("\nboth together:")
for v in ["exp_d","age_d"]:
    print(f"  {v:<6} {m.params[v]*100:+6.2f} pp   t {m.tvalues[v]:+5.1f}   p {m.pvalues[v]:.4f}")
print("\n-> experience strengthens, age flips NEGATIVE. They are not the same variable.")
print("   Caveat: at r=0.83 both coefficients are identified off ~30% of the")
print("   variance, so trust the signs more than the magnitudes.")

In [ ]:
# ----------------------------------------------------------------------
# R3: WHY AGE FLIPS -- DEBUT AGE IS A QUALITY SIGNAL
# ----------------------------------------------------------------------
deb = (sk.sort_values("yr").groupby("playerId").first()[["age"]]
         .rename(columns={"age": "debut_age"}))
Q = (player_careers.set_index("playerId").join(deb)
     .join(star_index.set_index("playerId")[["rel"]]).dropna(subset=["debut_age"]))
Q = Q[Q.debut_age.between(17, 30)]
print(f"n = {len(Q):,}")
print(f"corr(debut age, career span) = {Q.span.corr(Q.debut_age):+.3f}")
qq = Q.dropna(subset=["rel"])
print(f"corr(debut age, star index)  = {qq.rel.corr(qq.debut_age):+.3f}\n")
print(f"{'debut age':>12} {'n':>6} {'mean span':>11} {'star index':>12}")
for lo, hi in [(17,19),(20,20),(21,21),(22,22),(23,23),(24,26),(27,30)]:
    g = Q[Q.debut_age.round().between(lo, hi)]
    if len(g) > 40:
        lbl = f"{lo}" if lo == hi else f"{lo}-{hi}"
        print(f"{lbl:>12} {len(g):>6,} {g.span.mean():>11.2f} {g.rel.mean():>12.2f}")
print("\n-> teams fast-track players they rate. Arriving young is itself a")
print("   quality signal, which is why age-at-fixed-experience predicts badly.")

In [ ]:
# ----------------------------------------------------------------------
# R4: IS THE RELATIONSHIP STABLE ACROSS ERAS, AND IS IT AN EXPANSION ARTIFACT?
# ----------------------------------------------------------------------
# New franchises are young and bad simultaneously, for reasons unrelated to
# experience helping. Dropping their first three seasons tests that.
TS["decade"] = (TS.yr // 10) * 10
est = TS[TS.franchise_age >= 3]
print(f"{'decade':>8} {'n':>5} {'all teams':>11} {'excl. new franchises':>22}")
for dec, g in TS.groupby("decade"):
    if len(g) < 20: continue
    m1 = sm.OLS(g.point_pct_dev if "point_pct_dev" in g else g.point_pct_d,
                sm.add_constant(g[["exp_d"]])).fit()
    x = est[est.decade == dec]
    m2 = sm.OLS(x.point_pct_d, sm.add_constant(x[["exp_d"]])).fit()
    print(f"{int(dec)}s {len(g):>5} {m1.params.exp_d*100:>+10.2f} "
          f"{m2.params.exp_d*100:>+21.2f}")
print("\n-> positive in most decades but NEGATIVE in the 1930s, and it gets more")
print("   negative when new franchises are excluded. 'Experience wins' is a")
print("   modern regularity, not a law of the sport.")

In [ ]:
# ----------------------------------------------------------------------
# R5: SURVIVOR BIAS -- WHY NAIVE AGING CURVES LIE
# ----------------------------------------------------------------------
# The single most important methodological point in the project. Reproduced
# here because it is the reason several early versions of this analysis were
# wrong. Method after Tango; hockey adaptation after EvolvingWild (2017).
sks = read("skater_seasons").merge(skater_bios[["playerId","birth_date"]], on="playerId")
sks["yr"] = sks.season // 10000
sks["birth"] = pd.to_datetime(sks.birth_date, errors="coerce")
sks["age"] = ((pd.to_datetime(sks.yr.astype(str) + "-10-01") - sks.birth).dt.days/365.25).round()
sks = sks[sks.age.between(18,40) & sks.games_played.ge(20)].copy()
sks["age"] = sks.age.astype(int)
sks["ppg"] = sks.points / sks.games_played
sks = sks.merge(star_index[["playerId","tier"]], on="playerId").sort_values(["playerId","yr"])
sks["cs"] = sks.groupby("playerId").cumcount() + 1
# tiers are assigned on seasons 1-3, so measuring aging inside that window is circular
sks = sks[sks.cs >= 4]
g = sks.groupby("playerId")
sks["d_ppg"] = sks.ppg - g.ppg.shift(1)
sks["d_age"] = sks.age - g.age.shift(1)

TIERS = ["Regular","Solid","Star","Generational"]
pk = sks[sks.age.between(22,25)].groupby("playerId").ppg.mean().rename("pk")
ix = sks.join(pk, on="playerId").dropna(subset=["pk"]); ix = ix[ix.pk > 0]
ix["idx"] = ix.ppg / ix.pk
naive = ix.groupby([pd.cut(ix.age,[25,28,31,34,42]), "tier"], observed=True).idx.mean().unstack()[TIERS]

base = sks[sks.age.between(24,26)].groupby("tier", observed=True).ppg.mean()
dl = sks[sks.d_age.eq(1)]
delta = (dl.groupby([pd.cut(dl.age,[21,24,27,30,33,42]), "tier"], observed=True)
           .d_ppg.mean().unstack()[TIERS] / base * 100)

print("NAIVE: points per game as a share of the player's own early-career peak")
print(naive.round(3).to_string())
print("\nDELTA METHOD: same players, paired consecutive seasons, % of own peak")
print(delta.round(1).to_string())
print("\n-> The naive curve says Regulars age BEST. They do not. Only the")
print("   Regulars who improved are still in the league to be measured.")
print("   Pairing each player with himself a year later reverses the ranking.")

## Summary of Key Findings

### 1. **The league really is more veteran than ever**
- 31% of players are in their tenth season or later, the highest on record
- Rookies are 12% of the league, near an all-time low
- The only comparable season is 1966-67, when the league had six teams
- The claim is threshold-sensitive: on an 11th-season definition, 1966-67 still leads

### 2. **Careers are not getting longer**
- No trend in median or mean career length across the century
- The 1960s produced the longest careers, the 1970s the shortest
- Variation is cyclical and tracks expansion, which floods the league with
  marginal players who wash out quickly

### 3. **The composition shift is mostly arithmetic**
- Frozen careers plus real intake reproduces the observed veteran share to
  within a point or two through 2020
- A population that admits proportionally fewer new members ages, regardless
  of how long anyone stays
- Roughly half of today's record is intake, half is veterans becoming less
  likely to leave

### 4. **Experience marks a good roster, not a good night**
- Between teams: about +3 points of points percentage per season of experience
- Within a team-season: zero, with a tight interval
- Teams that made the playoffs were meaningfully more experienced than teams
  that missed
- Between the two teams that reached the final: a coin flip

### 5. **Experience is not age**
- Correlated at 0.83 within team-seasons but not interchangeable
- With both in the model, experience strengthens and age turns negative
- Debut age is itself a quality signal: players debuting at 17-19 average
  10.7-season careers against 3.0 for those debuting at 27+

### 6. **No evidence for**
- Careers lengthening over the century
- Rookies arriving younger (debut age has been flat at ~22 for seventy years)
- Expansion explaining the modern intake decline (correlation with
  years-since-expansion is ~0)
- Lineup experience affecting a given game's outcome
- The more experienced team winning the Stanley Cup
- A uniform historical relationship: the 1930s slope is significantly negative

### 7. **Known limitations**
- No phantom-season or imputation correction for survivor bias, so the aging
  estimates are conservative in a known direction (Lichtman; Nguyen & Matthews 2024)
- Points per game is not usage-adjusted; a veteran moved off the power play
  reads as decline
- Goalies are excluded from the star index and from all aging analysis
- Reverse causation is unaddressed: rebuilding teams play rookies *because*
  they are rebuilding
- Cohorts debuting after ~2010 cannot yet have their career length estimated